# XRF55 Bench — Ablation Ladder: TF-Mamba → WavDualMamba

Đi từ TF-Mamba (S0) đến WavDualMamba (S6), **mỗi bậc đổi đúng 1 thứ**.
Tổng các Δ giữa các bậc liên tiếp = đúng gap giữa 2 baseline.

Thứ tự thêm khối: **CNN → BiMamba → AttnStatPool** (front-to-back theo kiến trúc:
embedding → sequence → pooling). `kwargs` ở Cell 3 là **tích luỹ** — bậc sau gồm
mọi flag của bậc trước.

| Bậc | Thay đổi (so với bậc trước) | Flag cộng thêm | Trạng thái |
|---|---|---|---|
| **S0** | TF-Mamba baseline (haar {HL,LH}) | — | ✅ đã có (proc 89.16%, raw 86.39%) |
| **S1** | + CNN front-end WavDualMamba (kernel per-subband (3,7)/(7,3) + dilated TFBlocks) | `use_cnn=True` | chạy ở đây |
| **S2** | + gated BiMamba×2 d_state=32 (thay uni×3 d_state=16) | `+ mamba='bi'` | chạy ở đây |
| **S3** | + AttnStatPool (GAP → attentive mean+std) | `+ pool='attnstat'` | chạy ở đây |
| **S4** | dọn chi tiết (bỏ PE/proj_s3, fusion convex zero-init, head WDM) ≡ **WavDualMamba trên haar** | `wavdualmamba_haar` | chạy ở đây |
| **S5** | haar → db4 (kiến trúc y nguyên) | `wavdualmamba` + `subbands=('HL','LH')` | chạy ở đây |
| **S6** | + branch LL = WavDualMamba đầy đủ | `wavdualmamba` | ✅ đã có (proc 91.87%, raw 89.38%) |

Δ đọc theo bậc: **ΔS1** = CNN (trên baseline trần) · **ΔS2** = BiMamba (khi đã có
CNN) · **ΔS3** = AttnStatPool (khi đã có CNN+BiMamba). S3 = đủ cả 3 khối.

**Chuẩn mọi run:** protocol 02* (AdamW lr=5e-4, wd=1e-3, betas=(0.9,0.95),
warmup 10ep + cosine→1e-6, 80 epochs, bs=32, grad_clip=1.0), 5 seeds
{0,4,8,17,42}, eval `last_model.pt` — y hệt các run benchmark headline.

**Cách dùng:** chọn `RUNGS` ở Cell 3 (vd `['S1']` hoặc `['S1','S2']`), Run All.
Không sửa kwargs tay — `LADDER` là nguồn sự thật duy nhất.

**Lưu ý:**
- Push code mới nhất lên GitHub trước khi chạy (Cell 2 pull repo).
- ΔS6 kèm +~50% params (thêm nguyên branch LL) — ghi chú khi trình bày.
- S1–S3 dùng kernel WavDualMamba thật ((3,7)/(7,3)) — `subband_kernels=True` mặc
  định; nhờ vậy ΔS4 chỉ còn: bỏ PE/proj_s3, fusion zero-init, head WDM.
- Data Kaggle hiện build trước fix tráo nhãn cH/cV: cả PreprocTFMambaDataset
  (S0–S3) lẫn adapter S4 tự nhận diện qua `stats.meta.tfmamba_subband_naming`,
  đưa nội dung HL về stream_T (→ kernel (3,7)).

## Attached dataset

| Dataset name | Mount path |
|---|---|
| `xrf55-amp-dataset` | `/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/` |

In [ ]:
# Cell 1 — Install mamba-ssm (required by both model families)
!pip install -q ninja packaging wheel
!pip install -q triton
!pip install -q causal-conv1d>=1.2.0 --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation
print('Install done')

In [ ]:
# Cell 2 — Clone / update latest code from GitHub
import sys, subprocess
from pathlib import Path

CODE_PATH = Path('/kaggle/working/xrf55_benchmark')
if not CODE_PATH.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/imhoangt/xrf55_benchmark.git',
         str(CODE_PATH)],
        check=True
    )
else:
    subprocess.run(['git', 'pull'], cwd=str(CODE_PATH), check=True)

sys.path.insert(0, str(CODE_PATH))
print(f'Code path  : {CODE_PATH}')

from xrf55_bench.trainer import run
print('Import OK  : xrf55_bench.trainer.run')

In [ ]:
# Cell 3 — Ladder configuration: pick RUNGS, never edit kwargs per-run
from pathlib import Path

SEEDS      = [0, 4, 8, 17, 42]    # benchmark standard: full 5 seeds
RUNGS      = ['S5']               # rungs to run this session, e.g. ['S1', 'S2']
DATA_MODES = ['proc', 'raw']      # default: both

DATA_DIRS = {
    'raw':  Path('/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/raw'),
    'proc': Path('/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/processed'),
}

# ── The ladder — single source of truth (S0/S6 = existing headline runs) ──────
# Order: CNN -> BiMamba -> AttnStatPool. kwargs are CUMULATIVE (each rung adds
# one flag on top of the previous). S3 = all three blocks.
LADDER = {
    'S1': dict(model='tfmamba',
               kwargs={'use_cnn': True},
               note='S0 + WavDualMamba CNN front-end (SubbandStem + dilated TFBlocks)'),
    'S2': dict(model='tfmamba',
               kwargs={'use_cnn': True, 'mamba': 'bi'},
               note='S1 + gated BiMamba x2 d_state=32 (replaces uni-Mamba x3 d_state=16)'),
    'S3': dict(model='tfmamba',
               kwargs={'use_cnn': True, 'mamba': 'bi', 'pool': 'attnstat'},
               note='S2 + AttnStatPool (GAP -> attentive mean+std); = all three blocks'),
    'S4': dict(model='wavdualmamba_haar',
               kwargs={},
               note='full WavDualMamba arch on Haar {HL,LH} (tfmamba arrays, packed adapter)'),
    'S5': dict(model='wavdualmamba',
               kwargs={'subbands': ('HL', 'LH')},
               note='same arch, db4 {HL,LH} - wavelet swap (S6 = +LL = headline)'),
}

for r in RUNGS:
    assert r in LADDER, f'Unknown rung {r!r} - choose from {list(LADDER)}'
    print(f"{r}: model={LADDER[r]['model']:<18} kwargs={LADDER[r]['kwargs']}")
    print(f"    {LADDER[r]['note']}")
print(f'Seeds      : {SEEDS}')
print(f'Data modes : {DATA_MODES}')
for dm in DATA_MODES:
    p = DATA_DIRS[dm] / 'stats.json'
    print(f"  [{'OK' if p.exists() else 'MISSING'}] {p}")

In [ ]:
# Cell 4 — Run the selected rungs (protocol 02*, identical to all benchmark runs)
from xrf55_bench.config import TrainCfg_for_protocol

PROTOCOL    = '02'
OUTPUT_ROOT = Path('/kaggle/working/outputs')
RUN_DIRS    = []

for rung in RUNGS:
    spec = LADDER[rung]
    for dm in DATA_MODES:
        out = OUTPUT_ROOT / f'abl_{rung.lower()}_p{PROTOCOL}_{dm}'
        cfg = TrainCfg_for_protocol(
            PROTOCOL, seeds=tuple(SEEDS),
            num_epochs=80, betas=(0.9, 0.95), grad_clip=1.0,
            lr=5e-4, floor_lr=1e-6,
        )  # = protocol 02* — same as every benchmark headline run
        print('\n' + '#' * 70)
        print(f'##  {rung} [{dm}]  ->  {out}')
        print(f'##  {spec["note"]}')
        print('#' * 70)
        run(model_name=spec['model'], bench_dir=DATA_DIRS[dm], output_dir=out,
            train_cfg=cfg, num_workers=4, model_kwargs=spec['kwargs'])
        RUN_DIRS.append(out)

print('\nDone:', [d.name for d in RUN_DIRS])

In [ ]:
# Cell 5 — Per-run results (this session)
import json

for out in RUN_DIRS:
    p = out / 'metrics.json'
    if not p.exists():
        print(f'[MISSING] {p}')
        continue
    with open(p) as f:
        m = json.load(f)
    s = m['summary']
    print(f"{out.name:<24} acc={s['test_accuracy_mean']*100:6.2f}% ± {s['test_accuracy_std']*100:4.2f}%   "
          f"f1={s['test_f1_macro_mean']*100:6.2f}% ± {s['test_f1_macro_std']*100:4.2f}%   "
          f"params={s['params_M']}M   lat={s['latency_mean_ms']}ms")

In [ ]:
# Cell 6 — Plots + download zips
import shutil
from IPython.display import Image, display, FileLink

for out in RUN_DIRS:
    print(f'--- {out.name} ---')
    for fname in ['training_curve.png', 'confusion_matrix.png', 'seed_comparison.png']:
        p = out / 'plots' / fname
        if p.exists():
            display(Image(str(p)))
    for src in sorted(out.glob('*.zip')):
        dst = Path('/kaggle/working') / src.name
        if src.resolve() != dst.resolve():
            shutil.copy2(src, dst)
        print(f'{src.name}  ({dst.stat().st_size / 1e6:.1f} MB)')
        display(FileLink(dst.name))

In [ ]:
# Cell 7 — Ladder summary table (works on Kaggle or locally after collecting zips)
# Scans RESULT_ROOTS for abl_* metrics.json; S0/S6 use the headline numbers below.
import json
from pathlib import Path

RESULT_ROOTS = [Path('/kaggle/working/outputs'),
                Path('../../outputs/kaggle_plots')]   # local layout

# Headline baselines (5 seeds, protocol 02*) — the two ladder endpoints.
BASELINES = {
    ('S0', 'proc'): dict(acc=0.891616, acc_std=0.006674, f1=0.890964, f1_std=0.006885, params=0.219),
    ('S0', 'raw'):  dict(acc=0.863940, acc_std=0.004787, f1=0.863887, f1_std=0.005055, params=0.219),
    ('S6', 'proc'): dict(acc=0.918687, acc_std=0.001720, f1=0.917708, f1_std=0.001792, params=0.588),
    ('S6', 'raw'):  dict(acc=0.893838, acc_std=0.002378, f1=0.892546, f1_std=0.002310, params=0.588),
}
RUNG_ORDER = ['S0', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6']
RUNG_DESC  = {
    'S0': 'TF-Mamba baseline (haar HL+LH)',
    'S1': '+ CNN front-end',
    'S2': '+ gated BiMamba',
    'S3': '+ AttnStatPool',
    'S4': '= WavDualMamba arch (haar)',
    'S5': 'haar -> db4',
    'S6': '+ LL = WavDualMamba (headline)',
}

def _load(p):
    with open(p) as f:
        s = json.load(f)['summary']
    return dict(acc=s['test_accuracy_mean'], acc_std=s['test_accuracy_std'],
                f1=s['test_f1_macro_mean'], f1_std=s['test_f1_macro_std'],
                params=s['params_M'])

def find_metrics(rung, dm):
    if (rung, dm) in BASELINES:
        return BASELINES[(rung, dm)]
    for root in RESULT_ROOTS:
        if not root.exists():
            continue
        hits = (sorted(root.glob(f'abl_{rung.lower()}_p02_{dm}*/metrics.json'))
                + sorted(root.glob(f'abl_{rung.lower()}_p02_{dm}*/results_summary/metrics.json')))
        if hits:
            return _load(hits[-1])   # newest if several (timestamped names sort)
    return None

for dm in ['proc', 'raw']:
    print(f'\n======== Ladder [{dm}] ========')
    print(f"{'Rung':<5} {'Change':<34} {'Accuracy':>16} {'dAcc':>7} {'F1 macro':>16} {'Params':>8}")
    prev = None
    for rung in RUNG_ORDER:
        m = find_metrics(rung, dm)
        if m is None:
            print(f'{rung:<5} {RUNG_DESC[rung]:<34} {"- not run -":>16}')
            prev = None   # break the delta chain at a gap
            continue
        d = f'{(m["acc"] - prev) * 100:+.2f}' if prev is not None else ''
        acc = f"{m['acc']*100:5.2f}% +- {m['acc_std']*100:4.2f}%"
        f1  = f"{m['f1']*100:5.2f}% +- {m['f1_std']*100:4.2f}%"
        print(f'{rung:<5} {RUNG_DESC[rung]:<34} {acc:>16} {d:>7} {f1:>16} {m["params"]:>7.3f}M')
        prev = m['acc']
    print('Note: dS6 includes +~50% params (extra LL branch backbone).')